# LD-1 — FUMÉE Kaggle CPU : D, tests lourds (argv), T/E/V sur mf_0000, vrai checkpoint

In [ ]:

import time
T_START = time.monotonic()          # §16 (3) : pris au tout début de la cellule
import datetime, faulthandler, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld1 start', _utc(), flush=True)
OUT = Path("/kaggle/working")
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld1-kit', '/kaggle/input/datasets/mabaku/babaorum-ld1-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
CKPT_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/logs/checkpoint_last.pt', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/logs/checkpoint_last.pt']
INIT_SHA = 'ef5e826803c347f73e49c83c0f21f61399f9ea06ee985905bd17644dff3844fa'
TIMEOUTS = {'D': 900, 'T': 600, 'E': 4500, 'V': 900, 'pytest_heavy': 5400}
LIMITS = {'deadline_s': 7200, 'v_reserve_s': 900, 'stop_margin_s': 120, 'e_reserve_s': 300}
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld1_line_detr.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
CKPT = resolve_file(CKPT_CANDIDATES)
assert INIT_SHA != "NON FIXÉ", "sha complet du checkpoint non fixé"
assert sha256(CKPT) == INIT_SHA, "checkpoint : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "train_line_detr.py"), "--phase", phase,
            "--root", str(PAGES), "--init-checkpoint", str(CKPT), "--device", "cpu",
            "--cache-dir", "/tmp/ld1_hf", *extra]

SMOKE = OUT / "ld1_smoke"
RUN = SMOKE / "ld1"
SMOKE.mkdir(parents=True, exist_ok=True)
HEAVY_ENV = dict(os.environ, LD1_HEAVY="1", LD1_PAGES_ROOT=str(PAGES), PYTHONPATH=str(KIT / "src") + os.pathsep + str(KIT / "scripts"))
HEAVY_ENV["HF_HUB_OFFLINE"] = "1"
def _smoke_argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "train_line_detr.py"), "--phase", phase, "--root", str(PAGES),
            "--init-checkpoint", str(CKPT), "--device", "cpu", "--out", str(RUN), "--pages", "mf_0000",
            "--cache-dir", str(SMOKE / "hf"), *extra]
STEPS = [
    {"name": "D", "argv": [sys.executable, str(KIT / "scripts" / "train_line_detr.py"), "--phase", "D",
                            "--out", str(RUN), "--cache-dir", str(SMOKE / "hf")], "timeout_s": TIMEOUTS["D"]},
    # §13 (4) : TOUT tests/test_line_detr.py, sans -k ; rc écrit dans un fichier, sans bloquer la suite
    {"name": "pytest_heavy", "argv": ["sh", "-c", "cd \"" + str(KIT) + "\" && " + sys.executable + " -m pytest -q -rs "
                                      "tests/test_line_detr.py > \"" + str(SMOKE) + "/ld1_pytest.txt\" 2>&1; "
                                      "echo $? > \"" + str(SMOKE) + "/ld1_pytest_rc.txt\"; exit 0"],
     "env": HEAVY_ENV, "timeout_s": TIMEOUTS["pytest_heavy"]},
    {"name": "T", "argv": _smoke_argv("T", ["--timing-steps", "3", "--max-steps", "20"]), "timeout_s": TIMEOUTS["T"]},
    {"name": "E", "argv": _smoke_argv("E", ["--max-steps", "20"]), "timeout_s": TIMEOUTS["E"]},
    {"name": "V", "argv": _smoke_argv("V", ["--diagnostic"]), "timeout_s": TIMEOUTS["V"]},
]
SUMMARY = {"kind": "SMOKE_KAGGLE_CPU_VRAI_CHECKPOINT", "init_sha256": INIT_SHA}
EXIT = None
try:
    EXIT = C.execute_pipeline(SMOKE, STEPS, heartbeat_interval_s=60.0)
finally:
    for n in (20,):                              # la fumée n'écrit qu'un checkpoint (pas 20) : effacé par chemin
        (RUN / f"ld1_step{n:06d}.pt").unlink(missing_ok=True)      # la fumée ne rapporte que du JSON
    SUMMARY["pipeline"] = {k: EXIT[k] for k in ("status", "step", "exit_code", "durations_s")} if EXIT else None
    for name in ("ld1_pytest.txt", "ld1_pytest_rc.txt"):
        f = SMOKE / name
        if f.is_file():
            SUMMARY[name] = f.read_text()[-6000:]
    for name in ("ld1_timing_manifest.json", "ld1_report.json", "ld1_train_log.jsonl"):
        f = RUN / name
        if f.is_file():
            SUMMARY[name] = f.read_text()[-20000:] if name.endswith(".jsonl") else json.loads(f.read_text())
    (OUT / "ld1_smoke_summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=str))
    shutil.rmtree(SMOKE / "hf", ignore_errors=True)
print("SMOKE", SUMMARY.get("pipeline"), flush=True)
